# AegisRAG — Merge Adapters & Export GGUF (Local Mac)

Merges the SFT and DPO LoRA/DoRA adapters into the base Qwen2.5-7B-Instruct,
converts to F16 GGUF, then quantizes to Q4_K_M (`aegis_final.gguf`).

**Run cells top to bottom.**

## Cell 1 — Install dependencies

In [10]:
import sys, subprocess

# Install into the active conda env (handles paths with spaces correctly)
pkgs = ['transformers', 'peft', 'accelerate', 'torch']
for pkg in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    print(f'  {pkg} ok')
print('All dependencies installed.')

  transformers ok
  peft ok
  accelerate ok
  torch ok
All dependencies installed.


## Cell 2 — Set paths

In [11]:
from pathlib import Path

REPO_ROOT   = Path('/Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG')
LLAMA_CPP   = REPO_ROOT / 'llama.cpp'

SFT_ADAPTER = REPO_ROOT / 'checkpoints' / 'generator_sft'
DPO_ADAPTER = REPO_ROOT / 'checkpoints' / 'dpo'
MERGED_DIR  = REPO_ROOT / 'checkpoints' / 'generator_merged_new'
F16_GGUF    = REPO_ROOT / 'checkpoints' / 'generator.F16.gguf'
FINAL_GGUF  = REPO_ROOT / 'checkpoints' / 'aegis_final.gguf'

for label, p in [('SFT adapter', SFT_ADAPTER), ('DPO adapter', DPO_ADAPTER)]:
    status = 'FOUND' if p.exists() else 'MISSING'
    print(f'{label}: {status} ({p})')

SFT adapter: FOUND (/Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator_sft)
DPO adapter: FOUND (/Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/dpo)


## Cell 3 — Check device

In [12]:
import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print('CUDA GPU:', torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    DEVICE = 'cpu'   # keep merge on CPU — MPS runs out of memory on 7B merges
    print('Apple Silicon detected — merge will run on CPU (safer for 7B)')
else:
    DEVICE = 'cpu'
    print('CPU only')

print('torch:', torch.__version__)

Apple Silicon detected — merge will run on CPU (safer for 7B)
torch: 2.6.0


## Cell 4 — Clone llama.cpp (skip if already present)

In [13]:
import subprocess, sys

if not LLAMA_CPP.exists():
    print('Cloning llama.cpp...')
    subprocess.run(['git', 'clone', 'https://github.com/ggerganov/llama.cpp.git', str(LLAMA_CPP)], check=True)
else:
    print('llama.cpp already present, pulling latest...')
    subprocess.run(['git', '-C', str(LLAMA_CPP), 'pull'], check=True)

# Use subprocess so the path with spaces is handled correctly (no shell splitting)
req_file = LLAMA_CPP / 'requirements.txt'
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)], check=True)
    print('llama.cpp Python requirements installed.')
else:
    print('No requirements.txt found — skipping.')
print('llama.cpp ready.')

llama.cpp already present, pulling latest...
Already up to date.
llama.cpp Python requirements installed.
llama.cpp ready.


## Cell 5 — Build llama-quantize (CMake)

In [14]:
import platform

BUILD_DIR    = LLAMA_CPP / 'build'
QUANTIZE_BIN = BUILD_DIR / 'bin' / 'llama-quantize'

if not QUANTIZE_BIN.exists():
    print('Building llama-quantize with CMake...')
    BUILD_DIR.mkdir(parents=True, exist_ok=True)

    # Metal acceleration on Apple Silicon, CPU-only elsewhere
    metal_flag = '-DGGML_METAL=ON' if platform.processor() == 'arm' else '-DGGML_METAL=OFF'

    subprocess.run([
        'cmake', '-S', str(LLAMA_CPP), '-B', str(BUILD_DIR),
        '-DCMAKE_BUILD_TYPE=Release',
        metal_flag,
        '-DLLAMA_BUILD_TESTS=OFF',
        '-DLLAMA_BUILD_EXAMPLES=OFF',
    ], check=True)

    subprocess.run([
        'cmake', '--build', str(BUILD_DIR),
        '--target', 'llama-quantize', '-j4'
    ], check=True)
else:
    print('llama-quantize already built.')

print('Binary:', QUANTIZE_BIN, '| exists:', QUANTIZE_BIN.exists())

llama-quantize already built.
Binary: /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/llama.cpp/build/bin/llama-quantize | exists: True


## Cell 6 — Load base model

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'

print('Loading base model (downloads ~15 GB on first run, uses cache after)...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,   # avoids needing accelerate's device_map
    trust_remote_code=True,
)
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
print('Base model loaded.')

Loading base model (downloads ~15 GB on first run, uses cache after)...


Loading weights: 100%|██████████| 339/339 [00:05<00:00, 61.58it/s] 


Base model loaded.


## Cell 7 — Merge SFT adapter

In [16]:
if SFT_ADAPTER.exists():
    print(f'Applying SFT adapter from {SFT_ADAPTER}...')
    model = PeftModel.from_pretrained(model, str(SFT_ADAPTER))
    model = model.merge_and_unload()
    print('SFT adapter merged.')
else:
    print(f'WARNING: SFT adapter not found at {SFT_ADAPTER} — skipping.')

Applying SFT adapter from /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator_sft...
SFT adapter merged.


## Cell 8 — Merge DPO adapter

In [17]:
if DPO_ADAPTER.exists():
    print(f'Applying DPO adapter from {DPO_ADAPTER}...')
    model = PeftModel.from_pretrained(model, str(DPO_ADAPTER))
    model = model.merge_and_unload()
    print('DPO adapter merged.')
else:
    print(f'WARNING: DPO adapter not found at {DPO_ADAPTER} — skipping.')

Applying DPO adapter from /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/dpo...
DPO adapter merged.


## Cell 9 — Save merged model

In [18]:
import gc

MERGED_DIR.mkdir(parents=True, exist_ok=True)
print(f'Saving to {MERGED_DIR}...')
model.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
print('Saved.')

del model
gc.collect()
print('Memory freed.')

Saving to /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator_merged_new...


Writing model shards: 100%|██████████| 1/1 [02:26<00:00, 146.10s/it]


Saved.
Memory freed.


## Cell 10 — Convert merged model to F16 GGUF

In [19]:
convert_script = LLAMA_CPP / 'convert_hf_to_gguf.py'
if not convert_script.exists():
    convert_script = LLAMA_CPP / 'convert.py'   # older llama.cpp
if not convert_script.exists():
    raise FileNotFoundError('No convert script found in llama.cpp — re-run Cell 4')

print(f'Converting {MERGED_DIR} -> {F16_GGUF}...')
result = subprocess.run([
    sys.executable, str(convert_script),
    str(MERGED_DIR),
    '--outfile', str(F16_GGUF),
    '--outtype', 'f16',
], check=True)

size_gb = F16_GGUF.stat().st_size / 1024**3
print(f'F16 GGUF ready: {F16_GGUF} ({size_gb:.1f} GB)')

Converting /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator_merged_new -> /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator.F16.gguf...


INFO:hf-to-gguf:Loading model: generator_merged_new
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,             torch.float16 --> F16, shape = {3584, 152064}
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {3584, 152064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float32 --> F16, shape = {18944, 3584}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float32 --> F16, shape = {3584, 18944}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float32 --> F16, shape = {3584, 18944}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {512}
INFO:hf-to-gguf:blk.0.attn_k.weight,       to

F16 GGUF ready: /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator.F16.gguf (14.2 GB)


## Cell 11 — Quantize F16 → Q4_K_M

In [20]:
QUANTIZE_BIN = LLAMA_CPP / 'build' / 'bin' / 'llama-quantize'
if not QUANTIZE_BIN.exists():
    raise FileNotFoundError(f'llama-quantize not found — re-run Cell 5')

print(f'Quantizing {F16_GGUF} -> {FINAL_GGUF} (Q4_K_M)...')
subprocess.run([str(QUANTIZE_BIN), str(F16_GGUF), str(FINAL_GGUF), 'Q4_K_M'], check=True)

size_gb = FINAL_GGUF.stat().st_size / 1024**3
print(f'Done! aegis_final.gguf ready: {FINAL_GGUF} ({size_gb:.1f} GB)')

Quantizing /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator.F16.gguf -> /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/aegis_final.gguf (Q4_K_M)...


llama_print_build_info: build = 8940 (78433f606)
llama_print_build_info: built with AppleClang 21.0.0.21000099 for Darwin arm64
main: quantizing '/Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator.F16.gguf' to '/Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/aegis_final.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 339 tensors from /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/generator.F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              


main: quantize time = 17364.26 ms
main:    total time = 17364.26 ms
Done! aegis_final.gguf ready: /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/aegis_final.gguf (4.4 GB)


llama_model_quantize_impl: model size  = 14526.27 MiB (16.00 BPW)
llama_model_quantize_impl: quant size  =  4460.45 MiB (4.91 BPW)


## Cell 12 — Cleanup intermediate files (optional)

In [22]:
import shutil

# Uncomment to free ~43 GB of intermediate files
shutil.rmtree(MERGED_DIR, ignore_errors=True)
F16_GGUF.unlink(missing_ok=True)
print('Cleaned up.')

print('Skipping cleanup. Uncomment lines above to free ~43 GB.')
print(f'aegis_final.gguf: {FINAL_GGUF}')

Cleaned up.
Skipping cleanup. Uncomment lines above to free ~43 GB.
aegis_final.gguf: /Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/checkpoints/aegis_final.gguf


python: can't open file '/Users/dhruvparmar/DAU/sem_2/DS 615/Project/AegisRAG/notebooks/run.py': [Errno 2] No such file or directory
